# 02 — Demo end-to-end trên **Google Colab + Google Drive**

Notebook này train mô hình Transformer trên Colab GPU, dữ liệu để trên Google Drive.
Mở bằng *extension Colab trong VS Code* hoặc trên colab.research.google.com.

**Luồng:** mount Drive → clone repo → (crawl hoặc dùng data có sẵn) → chỉ báo → train →
đánh giá → chọn Top-N → đóng gói `models/` + `final_merged.csv` để tải về chạy website.

## 0. Mount Drive, clone repo, cài đặt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
# ⚠️ Đổi URL cho đúng repo của bạn nếu khác
REPO_URL = "https://github.com/HoangTechCS-AIE/MachineLearning-HaUI.git"
if not os.path.exists("MachineLearning-HaUI"):
    !git clone $REPO_URL
%cd MachineLearning-HaUI
!pip install -q -r requirements.txt

In [ ]:
import sys; sys.path.insert(0, '.')
from pathlib import Path
from src.config import load_config

# Dùng cấu hình Colab (paths trỏ Google Drive). Sửa BASE trong file này nếu cần:
#   config/config.colab.yaml
cfg = load_config("config/config.colab.yaml")
for p in [cfg.paths.raw_dir, cfg.paths.processed_dir, cfg.paths.model_dir, cfg.paths.figures_dir]:
    Path(cfg.abs_path(p)).mkdir(parents=True, exist_ok=True)
print("raw  :", cfg.abs_path(cfg.paths.raw_dir))
print("model:", cfg.abs_path(cfg.paths.model_dir))
print("merged:", cfg.abs_path(cfg.paths.merged_file))

## 1. Chuẩn bị dữ liệu (chọn 1 trong 2)

- **Cách A — crawl ngay trên Colab** (cần tài khoản FiinQuant): bỏ comment ô dưới.
- **Cách B — đã đẩy data lên Drive**: bỏ qua ô crawl, chỉ cần có `final_merged.csv`
  (hoặc các CSV thô trong `raw_dir`) trên Drive.

In [ ]:
# === Cách A: crawl trên Colab (cần tài khoản FiinQuant) ===
# import os
# os.environ["FIINQUANT_USERNAME"] = "TEN_DANG_NHAP"
# os.environ["FIINQUANT_PASSWORD"] = "MAT_KHAU"
# !pip install -q --extra-index-url https://fiinquant.github.io/fiinquantx/simple fiinquantx
# from src.data.crawl import crawl_all
# crawl_all(cfg)

### Cách C — dùng **dataset DSCT có sẵn trên Drive** (không cần FiinQuant)

Schema DSCT khớp pipeline này (cùng tên cột `timestamp/ticker/OHLCV` và chỉ báo).
**Yêu cầu:** folder `dataset` phải nằm trong *My Drive* — nếu đang ở *Shared with me*,
chuột phải folder → **Add shortcut to Drive** → My Drive.

Trỏ `raw_dir` vào folder DSCT rồi để `build_features` tính lại chỉ báo (kèm `vma20`) cho
đồng nhất. Nên dùng **Filler_Data** (OHLCV đã làm sạch); nếu thiếu, dùng `Raw_Data`.

In [ ]:
import glob, pandas as pd
# Đổi cho khớp đường dẫn Drive của bạn:
DSCT_DIR = "/content/drive/MyDrive/dataset/Filler_Data"
cfg.paths.raw_dir = DSCT_DIR

# Kiểm tra nhanh cấu trúc file (gửi output này cho mình nếu cần chỉnh)
files = sorted(glob.glob(DSCT_DIR + "/*.csv"))
print(f"{len(files)} file:", [f.split("/")[-1] for f in files][:6])
assert files, f'Không thấy CSV trong {DSCT_DIR} — kiểm tra đã add shortcut vào My Drive chưa.'
_sample = pd.read_csv(files[0], nrows=3)
print("Cột:", list(_sample.columns))
_sample

## 2. Tính chỉ báo kỹ thuật + gộp dữ liệu
Tạo `final_merged.csv` trên Drive (bỏ qua nếu đã có).

In [ ]:
from src.data.indicators import build_features
merged = Path(cfg.abs_path(cfg.paths.merged_file))
if merged.exists():
    print("Đã có", merged, "-> bỏ qua build_features")
else:
    build_features(cfg)

## 3. Train trên GPU

In [ ]:
import torch
print("CUDA khả dụng:", torch.cuda.is_available())
from src.train import train
out = train(cfg)
print("best val L1 =", out["best_val"])

## 4. Đánh giá trên tập test

In [ ]:
from src.evaluate import evaluate
res = evaluate(cfg, plot=True)
print("Transformer:", res["metrics"])
print("Naive      :", res["naive"])

## 5. Chọn Top-N cổ phiếu

In [ ]:
from src.predict import predict_test, select_top_stocks
pred = predict_test(cfg)
months = sorted(pred["timestamp"].dt.strftime("%Y-%m").unique())
print("Các tháng test:", months)
select_top_stocks(pred, months[-1], cfg.select.top_n)

## 6. Đóng gói artifacts để tải về chạy **website**

Tải `btl_artifacts.zip` về máy, giải nén rồi đặt:
- thư mục `models/*` → `models/` của repo local
- `final_merged.csv` → `data/processed/` của repo local

Sau đó chạy backend (`uvicorn backend.app:app`) là website dùng được model thật.

In [ ]:
import shutil, os
os.makedirs("/content/export/data/processed", exist_ok=True)
shutil.copytree(cfg.abs_path(cfg.paths.model_dir), "/content/export/models", dirs_exist_ok=True)
shutil.copy(cfg.abs_path(cfg.paths.merged_file), "/content/export/data/processed/")
shutil.make_archive("/content/btl_artifacts", "zip", "/content/export")
from google.colab import files
files.download("/content/btl_artifacts.zip")